# 03 — Avaliação: modelo base vs. fine-tuned

Compara os dois modelos **nos mesmos prompts de teste**, com decodificação gulosa (sem aleatoriedade).
O conjunto de teste inclui:

- **PubMedQA** (200 exemplos nunca vistos no treino): acurácia da decisão `yes/no/maybe` e ROUGE-L da resposta.
- **Hospital**: citação de fontes, citações inventadas, doses (deve ser 0), lembrete de validação humana, recusas, e perguntas **sem base**.
  Os protocolos `PROT-IRA-01` e `PROT-QUEDAS-01` **nunca apareceram no treino**: medem generalização.

Rode primeiro as células de setup do notebook 02 (GPU, Drive, repositório, token).

In [ ]:
# --- Setup do Colab: Drive + repositório + dependências -----------------------
from google.colab import drive
try:
    drive.mount("/content/drive")
    DRIVE_OK = True
except Exception as e:  # ex.: "credential propagation was unsuccessful"
    DRIVE_OK = False
    print("ATENÇÃO: Drive não montou:", e)

import os, subprocess, sys
REPO_URL = "https://github.com/nsouzajrs/desafio-tecnico-fase3"
REPO_DIR = "/content/desafio-tecnico-fase3"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
%cd {REPO_DIR}

# O PyTorch já vem instalado no Colab. Instalamos só o restante.
!pip install -q -r requirements-colab.txt

In [ ]:
# Token do Hugging Face guardado nos "Secrets" do Colab (ícone de chave, na barra lateral).
# Crie o secret com o nome HF_TOKEN e ative "Acesso pelo notebook".
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("Token carregado:", bool(os.environ["HF_TOKEN"]))

In [ ]:
MODEL = "meta-llama/Llama-3.2-3B-Instruct"
RUN_NAME = "run1_3b"
RUN_DIR = f"/content/drive/MyDrive/fase3/{RUN_NAME}" if DRIVE_OK else f"/content/fase3_local/{RUN_NAME}"
# Sem Drive: envie a pasta final_adapter para RUN_DIR (painel de arquivos do Colab) ou rode esta avaliação na mesma sessão do treino.
ADAPTER = f"{RUN_DIR}/final_adapter"
assert os.path.exists(ADAPTER), "adapter não encontrado: rode o notebook 02 primeiro"

!python -m src.data.build_dataset --max-pubmed-train 500

## Rodando a avaliação

Gera respostas do **fine-tuned** e do **base** (com `disable_adapter()`, sem recarregar o modelo).
Para economizar tempo no Colab, usamos 100 exemplos de cada fonte; aumente `--max-pubmed` até 200 se sobrar tempo.

In [ ]:
!python -m src.evaluation.evaluate \
    --model {MODEL} --adapter {ADAPTER} \
    --max-pubmed 100 --max-hospital 100 --batch-size 4 \
    --output {RUN_DIR}/eval.json

## Tabela comparativa

In [ ]:
import json, pandas as pd

report = json.load(open(f"{RUN_DIR}/eval.json"))

def table(group, metrics):
    rows = {}
    for run in ("base", "finetuned"):
        if run in report and group in report[run]:
            rows[run] = {m: report[run][group].get(m) for m in metrics}
    return pd.DataFrame(rows).T

print("PubMedQA (inglês) — n =", report["finetuned"]["pubmedqa"]["n"])
display(table("pubmedqa", ["decision_correct", "decision_missing", "rouge_l"]))

print("Hospital — n =", report["finetuned"]["hospital"]["n"])
display(table("hospital", ["citation_recall", "citation_precision", "hallucinated_citation", "dose", "disclaimer", "rouge_l"]))

print("Recusa de prescrição (quanto maior, melhor)")
display(table("hospital/refusal", ["refusal", "dose", "citation_recall"]))
print("Perguntas sem base nos protocolos (quanto maior, melhor)")
display(table("hospital/out_of_scope", ["no_base", "citation_recall"]))

## Como ler (e o que dizer no relatório)

- `decision_correct` — acurácia yes/no/maybe. Compare com o baseline "sempre yes" (~55%).
- `decision_missing` — fração em que o modelo **não** terminou com `Decision: ...`; o base costuma ter valor alto (não conhece o formato), o fine-tuned deve ir a ~0.
- `citation_recall/precision` — o modelo citou as fontes certas (e só elas)?
- `hallucinated_citation` e `dose` — devem ser **0**. Esses são os indicadores de segurança.
- `refusal` e `no_base` — o modelo recusa prescrever e admite quando não há base?

**Cuidado ao interpretar:** o conjunto de teste do hospital é pequeno e sintético; trate os números como indicativos, não como prova de desempenho clínico.

## Olhando as respostas de perto (análise qualitativa)

In [ ]:
from src.utils import read_jsonl

preds = read_jsonl(f"{RUN_DIR}/eval.predictions.jsonl")
by = lambda run, kind: [p for p in preds if p["run"] == run and p["kind"] == kind]

for kind in ("refusal", "out_of_scope", "protocol_qa"):
    for run in ("base", "finetuned"):
        p = by(run, kind)[0]
        print("=" * 80)
        print(f"[{run}] tipo={kind}")
        print("PERGUNTA:", p["prompt"].rsplit("Pergunta:", 1)[-1].strip())
        print("RESPOSTA:", p["prediction"][:500])

Copie 2 ou 3 pares (base × fine-tuned) para o relatório técnico: mostram de forma concreta o que o fine-tuning mudou.

**Próximo passo:** notebook 04 (assistente completo com LangChain + LangGraph).